# Notebook 02 – Data Cleaning

This notebook focuses on cleaning and preparing the Telco Customer Churn dataset for exploratory data analysis and machine learning.

The data quality assessment performed in Notebook 01 identified potential issues, particularly with the `TotalCharges` variable. This notebook addresses those issues while preserving the original customer-level records.

### Objectives

- Load the original dataset.
- Identify and handle missing or invalid values.
- Correct inappropriate data types.
- Check for duplicate records.
- Validate the cleaned dataset.
- Save the cleaned dataset for downstream analysis.

## Import Required Libraries

The following libraries are used for data manipulation, validation, and numerical operations during the cleaning process.

In [28]:
import pandas as pd
import numpy as np

## Load Dataset

The original Telco Customer Churn dataset is loaded as the starting point for the cleaning process. The raw dataset is retained separately so that the cleaning process remains reproducible.

In [29]:
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [30]:
df.shape

(7043, 21)

## Initial Data Quality Assessment

Before modifying the dataset, an initial quality assessment is performed to identify missing values, duplicate records, incorrect data types, and other potential data quality issues.

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [32]:
df.duplicated().sum()

np.int64(0)

In [33]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

## Initial Data Quality Assessment

Before applying any cleaning rules, the raw dataset is assessed using an SPSS-style variable layout. This provides a structured view of each variable's data type, number of valid records, missing values, and unique values.

In [34]:
quality_layout = pd.DataFrame({
    "Variable": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Valid": df.notna().sum().values,
    "Missing": df.isna().sum().values,
    "Unique": df.nunique().values
})

quality_layout

,Variable,Data Type,Valid,Missing,Unique
0,customerID,str,7043,0,7043
1,gender,str,7043,0,2
2,SeniorCitizen,int64,7043,0,2
3,Partner,str,7043,0,2
4,Dependents,str,7043,0,2
5,tenure,int64,7043,0,73
6,PhoneService,str,7043,0,2
7,MultipleLines,str,7043,0,3
8,InternetService,str,7043,0,3
9,OnlineSecurity,str,7043,0,3


In [35]:
quality_layout["Potential Issue"] = np.where(
    quality_layout["Missing"] > 0,
    "Missing Values",
    ""
)

quality_layout

,Variable,Data Type,Valid,Missing,Unique,Potential Issue
0,customerID,str,7043,0,7043,
1,gender,str,7043,0,2,
2,SeniorCitizen,int64,7043,0,2,
3,Partner,str,7043,0,2,
4,Dependents,str,7043,0,2,
5,tenure,int64,7043,0,73,
6,PhoneService,str,7043,0,2,
7,MultipleLines,str,7043,0,3,
8,InternetService,str,7043,0,3,
9,OnlineSecurity,str,7043,0,3,


## Investigating TotalCharges

The `TotalCharges` variable is stored as an object data type even though it represents a numerical billing amount. Before converting the column, the values are examined to determine whether non-numeric or blank entries are present.

In [36]:
df["TotalCharges"].unique()

<StringArray>
[  '29.85',  '1889.5',  '108.15', '1840.75',  '151.65',   '820.5',  '1949.4',
   '301.9', '3046.05', '3487.95',
 ...
 '2625.25', '6886.25',  '1495.1',   '743.3',  '1419.4',  '1990.5',  '7362.9',
  '346.45',   '306.6',  '6844.5']
Length: 6531, dtype: str

In [37]:
df["TotalCharges"].astype(str).str.strip().eq("").sum()

np.int64(11)

In [38]:
df.loc[
    df["TotalCharges"].astype(str).str.strip().eq(""),
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


## 📌 Business Observation

**Observation**

The investigation identified **11 records** where `TotalCharges` contains blank strings rather than explicit missing values. All 11 customers have a `tenure` of **0 months**, while `MonthlyCharges` is populated.

**Interpretation**

These records represent customers who appear to have recently started their service and therefore have not yet accumulated total charges.

**Cleaning Decision**

The blank `TotalCharges` values will be converted to numeric missing values during data cleaning and then treated as **0**, reflecting the absence of accumulated charges for customers with zero tenure.

**Important**

These customer records will be retained because they contain valid customer and service information and should not be removed simply because `TotalCharges` is blank.

In [39]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

## Convert TotalCharges to Numeric

The `TotalCharges` column represents a numerical billing amount but was originally stored as text. The column is converted to a numeric data type. Blank strings that cannot be converted are represented as missing values (`NaN`) for controlled handling.

In [40]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [41]:
df["TotalCharges"].dtype

dtype('float64')

In [42]:
df["TotalCharges"].isna().sum()

np.int64(11)

In [43]:
df.loc[
    df["TotalCharges"].isna(),
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,NaN,No
753,3115-CZMZD,0,20.25,NaN,No
936,5709-LVOEQ,0,80.85,NaN,No
1082,4367-NUYAO,0,25.75,NaN,No
1340,1371-DWPAZ,0,56.05,NaN,No
3331,7644-OMVMY,0,19.85,NaN,No
3826,3213-VVOLG,0,25.35,NaN,No
4380,2520-SGTTA,0,20.00,NaN,No
5218,2923-ARZLG,0,19.70,NaN,No
6670,4075-WKNIU,0,73.35,NaN,No


## Handling Missing TotalCharges

The 11 missing `TotalCharges` values belong exclusively to customers with zero tenure. Since these customers have not accumulated any total charges, the missing values are treated as zero rather than removing the records.

In [44]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [45]:
df["TotalCharges"].isna().sum()

np.int64(0)

In [46]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [47]:
df["TotalCharges"].isna().sum()

np.int64(0)

In [48]:
df["TotalCharges"].dtype

dtype('float64')

In [49]:
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [50]:
quality_layout = pd.DataFrame({
    "Variable": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Valid": df.notna().sum().values,
    "Missing": df.isna().sum().values,
    "Unique": df.nunique().values
})

quality_layout

,Variable,Data Type,Valid,Missing,Unique
0,customerID,str,7043,0,7043
1,gender,str,7043,0,2
2,SeniorCitizen,int64,7043,0,2
3,Partner,str,7043,0,2
4,Dependents,str,7043,0,2
5,tenure,int64,7043,0,73
6,PhoneService,str,7043,0,2
7,MultipleLines,str,7043,0,3
8,InternetService,str,7043,0,3
9,OnlineSecurity,str,7043,0,3


## 📌 Business Observation

**Observation**

The `TotalCharges` variable was successfully converted from text to numeric format. The 11 hidden blank values were identified, investigated, and treated as zero because all affected customers had zero tenure.

After cleaning, all 7,043 customer records contain valid `TotalCharges` values, with no missing values remaining.

**Data Integrity**

No customer records were removed during this cleaning step, preserving the original dataset size of 7,043 records.

**Next Step**

Perform a final data-quality validation to confirm that the cleaning process has not introduced any unintended changes.

## Duplicate Records Validation

After completing the data cleaning process, duplicate records are checked again to ensure that the cleaning operations did not introduce duplicate customer records.

Both complete-row duplicates and duplicate customer identifiers are validated.

In [51]:
duplicate_rows = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows}")

Duplicate rows: 0


In [52]:
duplicate_customers = df["customerID"].duplicated().sum()

print(f"Duplicate customer IDs: {duplicate_customers}")

Duplicate customer IDs: 0


In [53]:
print(f"Total records: {len(df)}")
print(f"Unique customers: {df['customerID'].nunique()}")

Total records: 7043
Unique customers: 7043


## 📌 Business Observation

**Observation**

No duplicate rows or duplicate customer identifiers were found after the cleaning process. The dataset contains 7,043 records representing 7,043 unique customers.

**Data Integrity**

The cleaning process preserved the one-record-per-customer structure of the original dataset and did not introduce duplicate records.

**Next Step**

Perform the final data-quality validation before saving the cleaned dataset.

## Final Data Quality Validation

A final validation is performed to confirm that the cleaning process has preserved the dataset structure and resolved the identified data quality issues. The validation checks record count, missing values, duplicate records, unique customer identifiers, and key data types.

In [54]:
print("FINAL DATA QUALITY CHECK")
print("-" * 40)

print(f"Dataset shape       : {df.shape}")
print(f"Missing values      : {df.isna().sum().sum()}")
print(f"Duplicate rows      : {df.duplicated().sum()}")
print(f"Unique customers    : {df['customerID'].nunique()}")
print(f"TotalCharges dtype  : {df['TotalCharges'].dtype}")
print(f"TotalCharges min    : {df['TotalCharges'].min()}")
print(f"TotalCharges max    : {df['TotalCharges'].max()}")

FINAL DATA QUALITY CHECK
----------------------------------------
Dataset shape       : (7043, 21)
Missing values      : 0
Duplicate rows      : 0
Unique customers    : 7043
TotalCharges dtype  : float64
TotalCharges min    : 0.0
TotalCharges max    : 8684.8


In [55]:
df["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

## 📌 Business Observation

**Validation Result**

The final quality assessment confirms that the cleaned dataset contains 7,043 customer records and 21 variables.

- No missing values remain.
- No duplicate records were identified.
- Each customer has a unique `customerID`.
- `TotalCharges` is stored as a numeric variable.
- The original customer population has been preserved.
- The `Churn` distribution remains unchanged.

**Conclusion**

The dataset has successfully passed the final data-quality validation and is ready for exploratory data analysis.

**Next Step**

Save the validated dataset in the processed data directory for use in subsequent notebooks.

In [57]:
df.to_csv("../data/processed/telco_customer_churn_clean.csv", index=False)

In [58]:
import os

os.path.exists("../data/processed/telco_customer_churn_clean.csv")

True

# Summary

### Key Findings

- The original dataset contained **7,043 customer records and 21 variables**.
- `TotalCharges` was incorrectly stored as a text variable.
- **11 hidden blank values** were identified in `TotalCharges`.
- All 11 affected customers had **zero tenure**.
- These values were treated as **0** rather than removing the records.
- `TotalCharges` was successfully converted to `float64`.
- No missing values remain after cleaning.
- No duplicate records or duplicate customer IDs were identified.
- All **7,043 customer records were preserved**.
- The target variable `Churn` remained unchanged: **5,174 No** and **1,869 Yes**.

### Conclusion

The dataset has successfully passed the data cleaning and validation process and is ready for exploratory data analysis.

The cleaned dataset has been saved separately from the raw source data to maintain reproducibility and data integrity.

### Next Notebook

**Notebook 03 – Exploratory Data Analysis (EDA)**